# JourneyGraph LLM Pipeline Demo

- Prerequisites: `.env` at the project root with `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` set. Neo4j running.
- Install: `uv sync --extra demo` (includes all LLM and notebook dependencies)

---
## Setup

Mirrors the startup sequence in `src/llm/run.py`:
1. `get_llm_config()` — hard-fails if `ANTHROPIC_API_KEY` is missing
2. `Neo4jManager()` — hard-fails if DB is unreachable
3. `SliceRegistry()` — validates YAML slices against live graph
4. `Planner()` — builds the LLM instance
5. `NarrationAgent()` — builds a second LLM instance with narration token budget

SliceRegistry validation completes before any LLM call to avoid wasting tokens.

In [10]:
import sys
from pathlib import Path

# Locate project root (where pyproject.toml lives) and add to sys.path.
# Walks up from the current working directory — works whether Jupyter is
# launched from the project root or from the demos/ subdirectory.
_cwd = Path().resolve()
_root = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "pyproject.toml").exists()),
    None,
)
if _root is None:
    raise RuntimeError("Could not find project root — no pyproject.toml in CWD or any parent.")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.common.config import get_llm_config
from src.common.logger import get_logger
from src.common.neo4j_tools import Neo4jManager
from src.llm.narration_agent import NarrationAgent
from src.llm.planner import Planner
from src.llm.slice_registry import SliceRegistry

log = get_logger(__name__)

llm_config = get_llm_config()
print(f"LLM config loaded — provider={llm_config.llm_provider}  "
      f"model={llm_config.llm_model}  max_tokens={llm_config.llm_max_tokens}")

db = Neo4jManager()
print("Neo4j connected")

registry = SliceRegistry(db, strict=False)
print(f"SliceRegistry loaded — domains: {registry.domains()}")

planner = Planner(registry, llm_config, strict=False)
print("Planner ready")

narration_agent = NarrationAgent(llm_config)
print("NarrationAgent ready")

LLM config loaded — provider=anthropic  model=claude-haiku-4-5-20251001  max_tokens=512
Neo4j connected
SliceRegistry loaded — domains: ['accessibility', 'delay_propagation', 'transfer_impact']
Planner ready
NarrationAgent ready


### Sample Questions

In [11]:
USER_QUERY = "How many trips were cancelled on the Red Line yesterday?"

---
## Step 1: Planner

Single LLM call that handles domain classification, path routing, and anchor extraction together:
- **domain** — `transfer_impact` / `delay_propagation` / `accessibility` / rejected
- **path** — `text2cypher` (precise counts/lookups), `subgraph` (topological context), or `both`
- **anchors** — stations, routes, dates, pathway nodes extracted from the query
- **use_gds** — whether Graph Data Science algorithms are warranted

JSON parse failure triggers one retry with a corrective nudge. On second failure, degrades to `text2cypher` with empty anchors and sets `parse_warning`.

In [12]:
planner_output = planner.run(USER_QUERY)

print(f"Query: {USER_QUERY!r}")
print("\u2550" * 56)
print(f"  domain           : {planner_output.domain!r}")
print(f"  path             : {planner_output.path!r}")
print(f"  schema_slice_key : {planner_output.schema_slice_key!r}")
print(f"  use_gds          : {planner_output.use_gds}")
print(f"  rejected         : {planner_output.rejected}")
print(f"  rejection_message: {planner_output.rejection_message!r}")
print(f"  path_reasoning   : {planner_output.path_reasoning!r}")
print(f"  anchor_notes     : {planner_output.anchor_notes!r}")
print(f"  parse_warning    : {planner_output.parse_warning}")
print("  anchors:")
print(f"    stations       : {planner_output.anchors.stations}")
print(f"    routes         : {planner_output.anchors.routes}")
print(f"    dates          : {planner_output.anchors.dates}")
print(f"    pathway_nodes  : {planner_output.anchors.pathway_nodes}")
print(f"    levels         : {planner_output.anchors.levels}")

if planner_output.rejected:
    print("\nQuery rejected — pipeline stops here.")

Query: 'How many trips were cancelled on the Red Line yesterday?'
════════════════════════════════════════════════════════
  domain           : 'transfer_impact'
  path             : 'text2cypher'
  schema_slice_key : 'transfer_impact'
  use_gds          : False
  rejected         : False
  rejection_message: None
  path_reasoning   : 'This is a specific count query requiring direct lookup of cancelled trips filtered by route and date.'
  anchor_notes     : "Route resolved as 'Red Line' from the query."
  anchors:
    stations       : []
    routes         : ['Red Line']
    dates          : ['yesterday']
    pathway_nodes  : []
    levels         : []


---
## Step 2: Anchor Resolution

Maps the Planner's anchor strings to Neo4j node IDs via full-text index lookup:
- Phase 1: candidate generation (full-text index, up to `candidate_limit` per mention)
- Phase 2: disambiguation (`TopK` by default, or `TypeWeightedCoherenceStrategy`)
- Relative date expressions (`"yesterday"`, `"last Tuesday"`) are normalised to `YYYYMMDD`

In [13]:
from datetime import UTC, datetime

from src.llm.anchor_resolver import AnchorResolver

CANDIDATE_LIMIT = 1   # 1 = baseline (no disambiguation). >1 enables strategy.
STRATEGY = "topk"     # "topk" or "coherence"

disambiguation_strategy = None
if STRATEGY == "coherence":
    from src.llm.disambiguation_strategies import TypeWeightedCoherenceStrategy
    disambiguation_strategy = TypeWeightedCoherenceStrategy()

invocation_time = datetime.now(UTC)

resolver = AnchorResolver(
    db=db,
    invocation_time=invocation_time,
    strategy=disambiguation_strategy,
    candidate_limit=CANDIDATE_LIMIT,
)
resolutions = resolver.resolve(planner_output.anchors)

print(f"Resolver config : {resolver.config}")
print(f"Any resolved?   {resolutions.any_resolved}")
print(f"Resolved flat:  {resolutions.as_flat_dict()}")
if resolutions.failed:
    print(f"Failed anchors: {resolutions.failed}")

if not resolutions.any_resolved:
    print("\nZero anchors resolved. Please restate with a specific station, route, or date.")

Resolver config : {'candidate_limit': 1, 'strategy': 'TopKStrategy'}
Any resolved?   True
Resolved flat:  {'Red Line': ['RED'], 'yesterday': ['20260411']}


---
## Step 3a: Subgraph Path

Runs only when `planner_output.path` is `"subgraph"` or `"both"`.

- **HopExpander** — bidirectional hop expansion from anchor nodes, constrained by `DomainExpansionConfig`
- **ContextSerializer** — serialises to a text block and enforces a 2 000-token budget (`tiktoken cl100k_base`)

In [14]:
from src.llm.subgraph_builder import SubgraphBuilder

subgraph_output = None

if planner_output.path in {"subgraph", "both"}:
    builder = SubgraphBuilder(db=db)
    subgraph_output = builder.run(
        planner_output,
        resolutions,
        resolver_config=resolver.config,
    )

    print("-" * 50)
    print(f"  domain            : {subgraph_output.domain!r}")
    print(f"  success           : {subgraph_output.success}")
    print(f"  failure_reason    : {subgraph_output.failure_reason!r}")
    print(f"  node_count        : {subgraph_output.node_count}")
    print(f"  trimmed           : {subgraph_output.trimmed}")
    print(f"  anchor_resolutions: {subgraph_output.anchor_resolutions}")
    print(f"  resolver_config   : {subgraph_output.resolver_config}")
    print(f"  provenance_nodes  : {len(subgraph_output.provenance_nodes)} node(s)")
    print("-" * 50)
    print(subgraph_output.context if subgraph_output.context else "  (empty)")
else:
    print(f"Subgraph path not selected (path={planner_output.path!r}) — skipping.")

Subgraph path not selected (path='text2cypher') — skipping.


---
## Step 3b: Text2Cypher Path

Runs only when `planner_output.path` is `"text2cypher"` or `"both"`.

**QueryWriter** (up to 3 attempts):
1. Loads `conventions.json` and `.cypher` few-shot examples for the domain
2. Builds a system prompt with the schema slice (node/rel whitelist, WMATA quirks) and resolved anchors
3. Makes a single LLM call to generate Cypher

**CypherValidator** (per attempt):
- Write-guard regex, blocked-CALL guard, `EXPLAIN` syntax check, label/rel/property whitelist
- Validation errors are fed back as targeted refinement hints on retry

In [15]:
from src.llm.cypher_validator import validate_and_log_cypher
from src.llm.query_writer import run_query_writer
from src.llm.text2cypher_output import Text2CypherOutput

_MAX_ATTEMPTS = 3
t2c_output = None

if planner_output.path in {"text2cypher", "both"}:
    schema_slice = registry.get(planner_output.schema_slice_key)
    refinement_errors: list[str] = []
    all_validation_notes: list[str] = []

    for attempt in range(1, _MAX_ATTEMPTS + 1):
        qw_output = run_query_writer(
            USER_QUERY,
            planner_output,
            llm_config,
            schema_slice=schema_slice,
            resolved_anchors=resolutions.as_flat_dict(),
            refinement_errors=refinement_errors or None,
            use_gds=planner_output.use_gds,
        )
        print(f"\n[Query Writer — attempt {attempt}/{_MAX_ATTEMPTS}]")
        print("Cypher:\n", qw_output.cypher_query)
        if attempt == 1:
            print("Chain-of-Thought:\n", qw_output.cot_comments)

        val_result = validate_and_log_cypher(
            qw_output.cypher_query,
            schema_slice,
            schema_slice.property_registry,
            db.driver,
            log,
        )

        if val_result.valid:
            print(f"Validator: valid (attempt {attempt}) — {val_result.results}")
            t2c_output = Text2CypherOutput(
                cypher=qw_output.cypher_query,
                results=val_result.results or [],
                domain=planner_output.domain,
                attempt_count=attempt,
                validation_notes=all_validation_notes,
                success=True,
            )
            break

        print(f"Validator: invalid — {val_result.errors}")
        all_validation_notes.extend(val_result.errors)
        refinement_errors = val_result.errors
    else:
        print(f"Validator: all {_MAX_ATTEMPTS} attempts failed.")
        t2c_output = Text2CypherOutput(
            cypher="",
            results=[],
            domain=planner_output.domain,
            attempt_count=_MAX_ATTEMPTS,
            validation_notes=all_validation_notes,
            success=False,
        )
else:
    print(f"Text2Cypher path not selected (path={planner_output.path!r}) — skipping.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: Cancellation)", position=<SummaryInputPosition line=1, column=52, offset=51>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'column': 52, 'offset': 51, 'line': 1}}> for query: "EXPLAIN MATCH (d:Date)<-[:ON_DATE]-(i:Interruption:Cancellation)-[:AFFECTS_ROUTE]->(r:Route:Rail)\nWHERE r.route_short_name = 'RED' AND d.date = '20260411'\nRETURN count(DISTINCT i) AS cancelled_trips, d.date AS date"
Received


[Query Writer — attempt 1/3]
Cypher:
 MATCH (d:Date)<-[:ON_DATE]-(i:Interruption:Cancellation)-[:AFFECTS_ROUTE]->(r:Route:Rail)
WHERE r.route_short_name = 'RED' AND d.date = '20260411'
RETURN count(DISTINCT i) AS cancelled_trips, d.date AS date
Chain-of-Thought:
 **Explanation:**

This query finds all cancellation interruptions on the Red Line for the specific date (yesterday, 20260411). It:
1. Matches cancellation interruptions linked to a date
2. Filters for the Red Line by route_short_name = 'RED' and the Rail label
3. Filters for the exact date '20260411'
4. Returns the count of distinct cancellations and the date

Since we have a resolved date anchor, we use a direct equality filter rather than the two-pass temporal pattern shown in the examples.
Validator: valid (attempt 1) — []


---
## Step 4: NarrationAgent

Terminal LLM call. Receives the Planner output, Text2Cypher results, and Subgraph context, then produces the final plain-English answer.

Response mode is selected by pure logic (no LLM) before the call:
- `synthesis` — both paths succeeded; lead with facts, explain pattern
- `precision` — Text2Cypher only; answer directly from counts
- `contextual` — Subgraph only; describe topology, qualify quantities
- `degraded` — neither succeeded; surface what failed and suggest rephrasing

In [16]:
narration_output = narration_agent.run(
    USER_QUERY,
    planner_output,
    t2c_output=t2c_output,
    subgraph_output=subgraph_output,
    resolutions=resolutions,
)

print(f"Mode    : {narration_output.mode}")
print(f"Sources : {narration_output.sources_used}")
print(f"Success : {narration_output.success}")
if not narration_output.success:
    print(f"Failure : {narration_output.failure_reason}")
print()
print("Answer")
print("\u2550" * 56)
print(narration_output.answer)

Mode    : precision
Sources : ['text2cypher']
Success : True

Answer
════════════════════════════════════════════════════════
I don't have data available for Red Line trip cancellations yesterday. The query returned no results.

To provide you with this information, I would need:
- The specific date you're asking about
- Access to cancellation records for that date

If you have a specific date in mind, please provide it and I can search again. Alternatively, you might find this information through:
- WMATA's service alerts page
- The official WMATA website's daily reports
- Customer service contact


---
## Agentic Pipeline — same query, dynamic tool selection

Runs `AgentOrchestrator` on `USER_QUERY` using the same Planner output and anchor resolutions from above.

Instead of the fixed QueryWriter → Validator → Subgraph fork, a Claude agent loop (max 5 iterations) selects tools dynamically:
- `full_text_search` — entity lookup via AnchorResolver
- `cypher_query` — QueryWriter + CypherValidator
- `subgraph_expand` — SubgraphBuilder hop expansion
- `entity_clarify` — AnchorClarifier repair pass

The same `NarrationAgent` produces the final answer. Compare mode, sources, and answer against the static result above.

In [17]:
from src.llm.agent import AgentOrchestrator
from src.llm.anchor_clarifier import AnchorClarifier

clarifier = AnchorClarifier(db, llm_config)

orchestrator = AgentOrchestrator(
    db=db,
    llm_config=llm_config,
    registry=registry,
    clarifier=clarifier,
    narration_agent=narration_agent,
)

agent_t2c, agent_sub, agent_narration = orchestrator.run(
    USER_QUERY,
    planner_output,
    resolutions,
    resolver,
    invocation_time,
)

print(f"Mode    : {agent_narration.mode}")
print(f"Sources : {agent_narration.sources_used}")
print(f"Success : {agent_narration.success}")
if not agent_narration.success:
    print(f"Failure : {agent_narration.failure_reason}")

if agent_narration.agent_trace:
    history = agent_narration.agent_trace.get("tool_call_history", [])
    print(f"Tools used ({len(history)} call(s)):")
    for call in history:
        print(f"  [{call.get('iteration', '?')}] {call.get('tool_name')} — {call.get('summary', '')}")

print()
print("Answer")
print("\u2550" * 56)
print(agent_narration.answer)

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description="One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: Cancellation)", position=<SummaryInputPosition line=1, column=52, offset=51>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'column': 52, 'offset': 51, 'line': 1}}> for query: "EXPLAIN MATCH (d:Date)<-[:ON_DATE]-(i:Interruption:Cancellation)-[:AFFECTS_ROUTE]->(r:Route:Rail)\nWHERE r.route_short_name = 'RED' AND d.date = '20260411'\nRETURN count(DISTINCT i) AS cancelled_trips, d.date AS date"
Received

Mode    : precision
Sources : ['text2cypher']
Success : True
Tools used (2 call(s)):
  [?] None — 
  [?] None — 

Answer
════════════════════════════════════════════════════════
I don't have data available to answer your question about Red Line trip cancellations yesterday.

The query returned no results. This could mean:

1. **No cancellations occurred** on the Red Line yesterday
2. **Data is not available** for the requested time period
3. **The specific query parameters** (date, line designation) don't match the available dataset

To get this information, I would need:
- Confirmation of the specific date you're asking about
- Access to WMATA's operational cancellation logs for that date

If you have additional context about the date or want to check a different time period, please let me know and I can try again.


In [18]:
db.close()